In [5]:
import codeStat.modelimports.modelFunction as func
from codeStat.modelimports.modelFunction import *
import librosa
import glob

D:\doc\2026\FINAL_CODE\src


In [8]:
dir_save_path="D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/GMM_TDNN/nonfix_out/GMM"

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI BANTU
# ─────────────────────────────────────────────────────────────────────────────

def baca_transkrip(wav_path: str) -> str:
    """Baca file transkrip (.txt) berpasangan dengan audio."""
    txt_path = wav_path.replace(".wav", ".txt")
    if not os.path.exists(txt_path):
        return ""
    with open(txt_path, "r", encoding="utf-8") as f:
        baris = f.read().strip()
    bagian = baris.split(None, 1)
    return bagian[-1].lower() if len(bagian) > 1 else bagian[0].lower()


# ─────────────────────────────────────────────────────────────────────────────
# KONFIGURASI
# ─────────────────────────────────────────────────────────────────────────────

sampled_root  = func.sampled_small_root    # ← ganti sesuai folder Anda
N_COMPONENTS  = 16                   # jumlah komponen Gaussian (K)
N_ITER        = 100                  # maksimum iterasi EM


# ─────────────────────────────────────────────────────────────────────────────
# TAHAP 1 — EKSTRAKSI FITUR MFCC
# ─────────────────────────────────────────────────────────────────────────────

hasil_mfcc  = []
X_per_class = {}   # {label: [array_mfcc, ...]}

for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
    try:
        audio_id = os.path.splitext(os.path.basename(wav_path))[0]

        # Ekstraksi fitur menggunakan fungsi milik Anda
        fitur = extract_features(wav_path)   # (T, 39) — MFCC + Δ + ΔΔ

        # Baca label transkrip
        label = baca_transkrip(wav_path)

        hasil_mfcc.append({
            "Id"            : audio_id,
            "Label"         : label,
            "Jumlah Frame"  : fitur.shape[0],
            "Dimensi Fitur" : fitur.shape[1],
            "Contoh MFCC[0]": fitur[0, :3].tolist() if len(fitur) > 0 else [],
        })

        # Kumpulkan per kelas untuk pelatihan GMM
        if label not in X_per_class:
            X_per_class[label] = []
        X_per_class[label].append(fitur)

    except Exception as e:
        print(f"Error pada file: {wav_path} : {e}")

df_mfcc = pd.DataFrame(hasil_mfcc)
df_mfcc.to_csv("HasilMFCC.csv", index=False)
print(f"HasilMFCC.csv = {len(df_mfcc)} audio")


# ─────────────────────────────────────────────────────────────────────────────
# TAHAP 2 — PELATIHAN GMM PER KELAS
# ─────────────────────────────────────────────────────────────────────────────

# Gabungkan semua bingkai per kelas
X_per_class = {
    label: np.vstack(frames)
    for label, frames in X_per_class.items()
    if len(frames) > 0
}

hasil_gmm_training = []
gmm_models         = {}   # {label: GaussianMixtureModel}

for label, X in X_per_class.items():
    try:
        gmm = GaussianMixtureModel(
            n_components=N_COMPONENTS,
            n_iter=N_ITER,
            tol=1e-4,
            reg_covar=1e-6,
            random_state=hash(label) % 10000
        )
        gmm.fit(X)
        gmm_models[label] = gmm

        hasil_gmm_training.append({
            "Label"               : label,
            "Jumlah Bingkai"      : X.shape[0],
            "Dimensi Fitur"       : X.shape[1],
            "K Komponen"          : N_COMPONENTS,
            "Iterasi EM"          : gmm.n_iter_done_,
            "Konvergen"           : gmm.converged_,
            "Log-Likelihood Akhir": round(gmm.log_likelihood_history_[-1], 6),
            "Bobot w_k"           : gmm.weights_.round(4).tolist(),
        })

    except Exception as e:
        print(f"Error pelatihan GMM label '{label}': {e}")

df_gmm_training = pd.DataFrame(hasil_gmm_training)
df_gmm_training.to_csv("HasilGMMTraining.csv", index=False)
print(f"HasilGMMTraining.csv = {len(df_gmm_training)} kelas")


# ─────────────────────────────────────────────────────────────────────────────
# TAHAP 3 — PREDIKSI & EVALUASI PER AUDIO
# ─────────────────────────────────────────────────────────────────────────────

hasil_prediksi   = []
hasil_gmm_detail = []

for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
    try:
        audio_id = os.path.splitext(os.path.basename(wav_path))[0]

        # Ekstraksi fitur menggunakan fungsi milik Anda
        fitur      = extract_features(wav_path)   # (T, 39)
        label_asli = baca_transkrip(wav_path)

        # Hitung skor log-likelihood ke setiap GMM
        skor_semua = {
            label: float(gmm.score(fitur))
            for label, gmm in gmm_models.items()
        }

        label_prediksi = max(skor_semua, key=skor_semua.get)
        benar          = (label_prediksi == label_asli)

        hasil_prediksi.append({
            "Id"            : audio_id,
            "Label Asli"    : label_asli,
            "Label Prediksi": label_prediksi,
            "Benar"         : benar,
            "Skor Terbaik"  : round(skor_semua[label_prediksi], 4),
        })

        for kelas, skor in skor_semua.items():
            hasil_gmm_detail.append({
                "Id"     : audio_id,
                "Kelas"  : kelas,
                "Skor"   : round(skor, 4),
                "Terbaik": (kelas == label_prediksi),
            })

    except Exception as e:
        print(f"Error prediksi file: {wav_path} : {e}")

df_prediksi   = pd.DataFrame(hasil_prediksi)
df_gmm_detail = pd.DataFrame(hasil_gmm_detail)

df_prediksi.to_csv(f'{dir_save_path}/prediksiGMM.csv', index=False)
df_gmm_detail.to_csv(f'{dir_save_path}/HasilGMMDetail.csv', index=False)

akurasi = df_prediksi["Benar"].mean() * 100 if len(df_prediksi) > 0 else 0
print(f"HasilPrediksiGMM.csv = {len(df_prediksi)} audio")
print(f"HasilGMMDetail.csv   = {len(df_gmm_detail)} baris")
print(f"Akurasi Klasifikasi  = {akurasi:.2f}%")

HasilMFCC.csv = 1749 audio
    Iterasi   1: log-likelihood = -142.729758  Δ = inf
    Iterasi  10: log-likelihood = -132.722020  Δ = 3.33e-02
    Iterasi  20: log-likelihood = -132.609024  Δ = 4.77e-03
    Iterasi  30: log-likelihood = -132.588898  Δ = 1.14e-03
    Iterasi  40: log-likelihood = -132.581635  Δ = 4.86e-04
    Iterasi  50: log-likelihood = -132.577815  Δ = 3.63e-04
    Iterasi  60: log-likelihood = -132.573490  Δ = 4.95e-04
    Iterasi  70: log-likelihood = -132.568485  Δ = 4.76e-04
    Iterasi  80: log-likelihood = -132.563860  Δ = 4.81e-04
    Iterasi  90: log-likelihood = -132.558226  Δ = 6.45e-04
    Iterasi 100: log-likelihood = -132.550833  Δ = 8.07e-04
    ⚠ Belum konvergen setelah 100 iterasi
HasilGMMTraining.csv = 1 kelas
HasilPrediksiGMM.csv = 1749 audio
HasilGMMDetail.csv   = 1749 baris
Akurasi Klasifikasi  = 100.00%


In [ ]:
import os, glob

sampled_root = func.sampled_small_root

for wav_path in list(glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True))[:5]:
    txt_path = wav_path.replace(".wav", ".txt")
    
    print(f"Audio : {wav_path}")
    print(f"TXT   : {txt_path}")
    print(f"Ada?  : {os.path.exists(txt_path)}")
    
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            print(f"Isi   : {f.read().strip()}")
    print()